In [25]:
import scanpy
import anndata
import matplotlib
from matplotlib import pyplot
import hdf5plugin
import numpy
import scvelo
import seaborn
import pandas
import warnings
import cellrank
import gseapy
import decoupler

In [26]:
# Read input file
working_directory = "RNA Sequencing Data/"

# adata = scanpy.read_h5ad(working_directory + "/velocity_sdevelo.h5ad")

# Or read saved anndata objects
base_name = "cellrank_1.3"
adata = scanpy.read_h5ad(
    working_directory+f"/{base_name} cells.h5ad"
)
dead_end_adata = scanpy.read_h5ad(
    working_directory+f"/{base_name} dead end genes.h5ad"
)
mESC_adata = scanpy.read_h5ad(
    working_directory+f"/{base_name} mESC genes.h5ad"
)
driver_df = pandas.read_csv(
    working_directory+f"{base_name} driver genes.csv",
)
full_driver_df = pandas.read_csv(
    working_directory+f"{base_name} raw driver genes.csv",
)

print(adata)

AnnData object with n_obs × n_vars = 12791 × 2000
    obs: 'barcode', 'batch', 'sample', 'group', 'day', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'n_counts', 'latent_time', 'sde_velocity_self_transition', 'manual_root', 'manual_end', 'sde_velocity_pseudotime', 'macrostates_fwd', 'term_states_fwd', 'term_states_fwd_probs', 'init_states_fwd', 'init_states_fwd_probs', 'clusters_gradients', 'fate_probabilities_Day 12 Control', 'fate_probabilities_mESC'
    var: 'ensemble_ids', 'gene_symbol', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'gene_count_corr', 'means', 'dispersions', 'dispersions_norm', 'highly_variable', 'fit_alpha', 'fit_beta', 'fit_gamma', 'fit_t_', 'fit_sigma_1', 'fit_sigma_2'
    uns: 'T_fwd_params', 'clusters_gradients_colors', 'coarse_fwd', 'eigendecomposition_fwd', 'init_states_fwd_colors', 'lineage_Day 12 Control_trend', 'lineage_mESC_t

In [5]:
# Show all possible libraries
names = gseapy.get_library_name(organism="Mouse")
print(names)

['ARCHS4_Cell-lines', 'ARCHS4_IDG_Coexp', 'ARCHS4_Kinases_Coexp', 'ARCHS4_TFs_Coexp', 'ARCHS4_Tissues', 'Achilles_fitness_decrease', 'Achilles_fitness_increase', 'Aging_Perturbations_from_GEO_down', 'Aging_Perturbations_from_GEO_up', 'Allen_Brain_Atlas_10x_scRNA_2021', 'Allen_Brain_Atlas_down', 'Allen_Brain_Atlas_up', 'Azimuth_2023', 'Azimuth_Cell_Types_2021', 'BioCarta_2013', 'BioCarta_2015', 'BioCarta_2016', 'BioPlanet_2019', 'BioPlex_2017', 'CCLE_Proteomics_2020', 'CM4AI_U2OS_Protein_Localization_Assemblies', 'COMPARTMENTS_Curated_2025', 'COMPARTMENTS_Experimental_2025', 'CORUM', 'COVID-19_Related_Gene_Sets', 'COVID-19_Related_Gene_Sets_2021', 'Cancer_Cell_Line_Encyclopedia', 'Carcinogenome', 'CellMarker_2024', 'CellMarker_Augmented_2021', 'ChEA_2013', 'ChEA_2015', 'ChEA_2016', 'ChEA_2022', 'Chromosome_Location', 'Chromosome_Location_hg19', 'ClinVar_2019', 'ClinVar_2025', 'DGIdb_Drug_Targets_2024', 'DSigDB', 'Data_Acquisition_Method_Most_Popular_Genes', 'DepMap_CRISPR_GeneDependency

## GO pathway

In [ ]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="GO_Biological_Process_2025",
    threads=16,
    min_size=5,
    max_size=1000,
    permutation_num=2000
)

2026-01-27 16:58:38,720 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [93]:
# Enriched in dead end

pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "ES", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,ES,NES,NOM p-val,FWER p-val,FDR q-val
0,Epidermis Development (GO:0008544),0.651482,1.832776,0.002571,0.4625,0.674046
1,Regulation of Trans-Synaptic Signaling (GO:009...,0.794153,1.80454,0.001188,0.591,0.505759
2,Regulation of Glycolytic Process (GO:0006110),0.816355,1.757079,0.004884,0.7985,0.661015
3,Proteolysis Involved in Protein Catabolic Proc...,0.743789,1.755606,0.006105,0.802,0.505198
7,Antigen Processing and Presentation of Exogeno...,0.858336,1.71164,0.010753,0.933,0.504058
5,Antigen Processing and Presentation of Peptide...,0.858336,1.71164,0.010753,0.933,0.504058
6,Antigen Processing and Presentation of Exogeno...,0.858336,1.71164,0.010753,0.933,0.504058
10,Negative Regulation of Myeloid Leukocyte Diffe...,0.761698,1.639885,0.025862,0.993,0.978322
11,Positive Regulation of Purine Nucleotide Catab...,0.871028,1.638279,0.009423,0.993,0.796093
12,Positive Regulation of Glycolytic Process (GO:...,0.871028,1.638279,0.009423,0.993,0.796093


In [94]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
4,Chromatin Organization (GO:0006325),-1.740139,0.003165,0.505,0.951183
8,Protein Polyubiquitination (GO:0000209),-1.69158,0.002494,0.7595,1.0
9,DNA Metabolic Process (GO:0006259),-1.652493,0.011735,0.9115,1.0
13,Male Meiotic Nuclear Division (GO:0007140),-1.636862,0.000833,0.9515,1.0
15,Transcription by RNA Polymerase II (GO:0006366),-1.622618,0.005882,0.971,1.0
16,Transcription Initiation-Coupled Chromatin Rem...,-1.600657,0.006071,0.9885,1.0
17,Regulation of Viral Genome Replication (GO:004...,-1.594757,0.017964,0.99,1.0
18,Regulation of Insulin Secretion (GO:0050796),-1.592842,0.008584,0.9915,1.0
24,Visual System Development (GO:0150063),-1.552395,0.006809,0.9995,1.0
26,Negative Regulation of Bone Remodeling (GO:004...,-1.538782,0.006843,1.0,1.0


## MSigDB_Hallmark_2020

In [ ]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='MSigDB_Hallmark_2020', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000,
)

2026-01-27 16:55:15,091 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [82]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
1,p53 Pathway,1.606322,0.002924,0.216,0.468555
2,Estrogen Response Early,1.567842,0.008197,0.279,0.31144
3,Cholesterol Homeostasis,1.501034,0.042184,0.433,0.36815
5,Androgen Response,1.439023,0.051213,0.549,0.407197
10,Allograft Rejection,1.373691,0.04058,0.707,0.49942
11,Estrogen Response Late,1.352284,0.043732,0.742,0.462977
12,Unfolded Protein Response,1.334377,0.142512,0.776,0.438806
13,IL-2/STAT5 Signaling,1.286984,0.059155,0.86,0.502488
15,mTORC1 Signaling,1.235123,0.136483,0.924,0.585074
16,Wnt-beta Catenin Signaling,1.231806,0.207254,0.926,0.533818


In [34]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,G2-M Checkpoint,-1.633575,0.006154,0.13,0.071303
4,Spermatogenesis,-1.468472,0.060914,0.583,0.227339
6,DNA Repair,-1.435013,0.052901,0.684,0.200886
7,Oxidative Phosphorylation,-1.399608,0.067395,0.786,0.206339
8,Mitotic Spindle,-1.384391,0.061934,0.822,0.184509
9,E2F Targets,-1.378908,0.078947,0.836,0.161083
14,Interferon Alpha Response,-1.272842,0.124611,0.97,0.276491
26,Adipogenesis,-1.022896,0.424961,1.0,0.800266
30,Bile Acid Metabolism,-0.923084,0.587931,1.0,0.987823
31,Interferon Gamma Response,-0.841091,0.777108,1.0,1.0


## ChipSeq targets, ChEA

In [35]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='ChEA_2022', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000
)

2026-01-27 16:10:48,957 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [36]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
24,RARG 19884340 ChIP-ChIP MEFs Mouse,1.723942,0.0,0.23,0.327244
26,JARID2 20075857 ChIP-Seq MESCs Mouse,1.705769,0.0,0.26,0.18913
29,SUZ12 18974828 ChIP-Seq MESCs Mouse,1.683325,0.0,0.309,0.157193
39,MTF2 20144788 ChIP-Seq MESCs Mouse,1.635858,0.0,0.441,0.18633
44,SUZ12 27294783 Chip-Seq ESCs Mouse,1.603422,0.0,0.551,0.203314
49,SUZ12 20075857 ChIP-Seq MESCs Mouse,1.54903,0.0,0.704,0.270837
51,TP63 17297297 ChIP-ChIP HaCaT Human,1.544842,0.007712,0.721,0.240856
54,PITX1 30713093 ChIP-Seq Epithelial Human Tongu...,1.535507,0.031941,0.745,0.228947
58,TP63 30713093 ChIP-Seq Epithelial Human Tongue...,1.501708,0.0,0.829,0.266275
62,JARID2 20064375 ChIP-Seq MESCs Mouse,1.496325,0.0,0.844,0.252339


In [37]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,FOXM1 23109430 ChIP-Seq U2OS Human,-2.080874,0.0,0.0,0.0
1,SMAD1 18555785 ChIP-Seq MESCs Mouse,-2.071709,0.0,0.0,0.0
2,TCF3 18692474 ChIP-Seq MESCs Mouse,-2.031104,0.0,0.0,0.0
3,MYBL2 22936984 ChIP-ChIP MESCs Mouse,-2.028161,0.0,0.0,0.0
4,NACC1 18358816 ChIP-ChIP MESCs Mouse,-2.023993,0.0,0.0,0.0
5,NANOG 18700969 ChIP-ChIP MESCs Mouse,-1.998363,0.0,0.0,0.0
6,NANOG 18555785 ChIP-Seq MESCs Mouse,-1.925791,0.0,0.004,0.000519
7,POU5F1 18358816 ChIP-ChIP MESCs Mouse,-1.924004,0.0,0.004,0.000454
8,SOX2 18692474 ChIP-Seq MESCs Mouse,-1.920891,0.0,0.005,0.000504
9,POU5F1 18700969 ChIP-ChIP MESCs Mouse,-1.909825,0.0,0.008,0.000726


## Enriched targets from ChipSeq, Ensembl

In [38]:
rank_data = driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='ENCODE_TF_ChIP-seq_2015', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000
)

2026-01-27 16:10:50,947 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [39]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
1,FOSL1 C2C12 mm9,1.958746,0.0,0.018,0.020186
7,SMARCC1 HeLa-S3 hg19,1.727731,0.0,0.194,0.125041
21,ZEB1 HepG2 hg19,1.635747,0.01039,0.407,0.213822
23,TAL1 G1E-ER4 mm9,1.616345,0.0,0.467,0.198496
25,ATF3 K562 hg19,1.570055,0.0,0.601,0.254343
31,EP300 ECC-1 hg19,1.543421,0.0,0.687,0.271576
42,SMARCC2 HeLa-S3 hg19,1.498263,0.016393,0.799,0.347647
46,MAX myocyte mm9,1.47301,0.009375,0.861,0.381852
48,RAD21 ECC-1 hg19,1.462778,0.010239,0.879,0.372444
51,JUND GM12878 hg19,1.447306,0.033766,0.908,0.382076


In [40]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,E2F4 MEL cell line mm9,-2.037136,0.0,0.0,0.0
2,FOXM1 ECC-1 hg19,-1.838486,0.0,0.029,0.014247
3,E2F4 CH12.LX mm9,-1.808197,0.0,0.045,0.01488
4,IRF3 GM12878 hg19,-1.804968,0.0,0.047,0.011635
5,E2F4 HeLa-S3 hg19,-1.739217,0.0,0.154,0.034383
6,SP2 K562 hg19,-1.735894,0.0,0.162,0.03071
8,NFYA GM12878 hg19,-1.717979,0.0,0.212,0.036499
9,NANOG H1-hESC hg19,-1.71797,0.00361,0.212,0.031937
10,FOS K562 hg19,-1.691803,0.0,0.286,0.041263
11,FOXM1 MCF-7 hg19,-1.69142,0.003205,0.287,0.037327


## GO pathways with full transcriptome

In [63]:
rank_data = full_driver_df[["gene_names", "Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="GO_Biological_Process_2025", # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000
)

2026-01-27 16:35:46,719 [WARNING] Duplicated values found in preranked stats: 1.89% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [64]:
# Enriched in dead end

pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "ES", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,ES,NES,NOM p-val,FWER p-val,FDR q-val
0,Ribosome Biogenesis (GO:0042254),0.727115,2.334127,0.0,0.0,0.0
1,Translation (GO:0006412),0.705754,2.321165,0.0,0.0,0.0
2,Ribonucleoprotein Complex Biogenesis (GO:0022613),0.737203,2.296186,0.0,0.0,0.0
3,Ribosomal Small Subunit Biogenesis (GO:0042274),0.769492,2.270756,0.0,0.0,0.0
4,rRNA Processing (GO:0006364),0.737638,2.258627,0.0,0.0,0.0
5,rRNA Metabolic Process (GO:0016072),0.709495,2.161552,0.0,0.0,0.0
6,Mitochondrial Translation (GO:0032543),0.697643,2.101657,0.0,0.0,0.0
7,Macromolecule Biosynthetic Process (GO:0009059),0.651389,2.101543,0.0,0.0,0.0
8,Mitochondrial Gene Expression (GO:0140053),0.695848,2.095677,0.0,0.0,0.0
9,Cytoplasmic Translation (GO:0002181),0.685243,2.093118,0.0,0.0,0.0


In [65]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
10,DNA-templated DNA Replication Maintenance of F...,-2.069062,0.0,0.007,0.006767
13,Male Meiotic Nuclear Division (GO:0007140),-1.971685,0.0,0.048,0.025619
14,Gonad Development (GO:0008406),-1.952828,0.0,0.088,0.030937
15,Development of Primary Male Sexual Characteris...,-1.942729,0.0,0.103,0.028036
16,Centromere Complex Assembly (GO:0034508),-1.941575,0.0,0.108,0.023589
17,Response to Estrogen (GO:0043627),-1.938539,0.0,0.116,0.021269
19,Meiosis I (GO:0007127),-1.906419,0.0,0.211,0.036323
20,Male Gonad Development (GO:0008584),-1.895446,0.0,0.261,0.040967
23,Male Gamete Generation (GO:0048232),-1.85393,0.0,0.485,0.080027
27,Meiotic Sister Chromatid Cohesion (GO:0051177),-1.842212,0.0,0.543,0.086913


## Enrichment analysis instead of ranking

In [41]:
driver_df.columns

Index(['Unnamed: 0', 'gene_names', 'Day 12 Control_corr',
       'Day 12 Control_pval', 'Day 12 Control_qval', 'Day 12 Control_ci_low',
       'Day 12 Control_ci_high', 'mESC_corr', 'mESC_pval', 'mESC_qval',
       'mESC_ci_low', 'mESC_ci_high'],
      dtype='object')

In [47]:
print(driver_df["Day 12 Control_pval"])

0       0.0
1       0.0
2       0.0
3       0.0
4       0.0
       ... 
1995    0.0
1996    0.0
1997    NaN
1998    NaN
1999    NaN
Name: Day 12 Control_pval, Length: 2000, dtype: float64


In [48]:
driver_df["Day 12 Control_pval"] < 0.05

0        True
1        True
2        True
3        True
4        True
        ...  
1995     True
1996     True
1997    False
1998    False
1999    False
Name: Day 12 Control_pval, Length: 2000, dtype: bool

In [79]:
dead_end_drivers = driver_df.sort_values("Day 12 Control_corr").head(100)["gene_names"].to_list()
background_genes = adata.var_names.tolist()

# Run Enrichr (ORA)
enr = gseapy.enrichr(
    gene_list=dead_end_drivers,
    gene_sets='GO_Biological_Process_2025',
    background=background_genes,
    organism='mouse',
    outdir=None
)

# Print results
enr.res2d.sort_values('Adjusted P-value').head(10)

,Gene_set,Term,P-value,Adjusted P-value,Old P-value,Old adjusted P-value,Odds Ratio,Combined Score,Genes
0,GO_Biological_Process_2025,Chromatin Organization (GO:0006325),0.000016,0.014716,0,0,9.631714,106.500213,TCF7L1;REST;PHC1;HMGB2;TET1;SYCP3;HMGN2;JARID2
1,GO_Biological_Process_2025,Chromatin Remodeling (GO:0006338),0.000795,0.330041,0,0,9.038278,64.504069,REST;PHC1;MTF2;TET1;JARID2
2,GO_Biological_Process_2025,Male Meiotic Nuclear Division (GO:0007140),0.001128,0.330041,0,0,29.350515,199.217395,DNMT3L;TDRD12;SYCP3
3,GO_Biological_Process_2025,ATP Metabolic Process (GO:0046034),0.002174,0.330041,0,0,19.556701,119.908460,UCP2;ENPP3;ATP1B1
4,GO_Biological_Process_2025,V(D)J Recombination (GO:0033151),0.002476,0.330041,0,0,inf,inf,TCF7L1;HMGB2
5,GO_Biological_Process_2025,Positive Regulation of Amyloid-Beta Clearance ...,0.002476,0.330041,0,0,inf,inf,LRPAP1;APOE
6,GO_Biological_Process_2025,Somatic Recombination of Immunoglobulin Gene S...,0.002476,0.330041,0,0,inf,inf,MSH6;TCF7L1
7,GO_Biological_Process_2025,Protein Polyubiquitination (GO:0000209),0.002969,0.346221,0,0,8.754630,50.948801,ARRDC4;TRIM71;IFI27;NEDD4
240,GO_Biological_Process_2025,Positive Regulation of Neuron Differentiation ...,0.101458,0.381467,0,0,4.287982,9.811376,TCF7L1;REST
239,GO_Biological_Process_2025,Unsaturated Fatty Acid Metabolic Process (GO:0...,0.101458,0.381467,0,0,4.287982,9.811376,GSTM2;GPX4


Less informative than using ranking

## Enriched in clusters

In [18]:
# Enriched among mid_peak_genes
mid_peak_genes = dead_end_adata[dead_end_adata.obs["clusters"] == "1"]


In [19]:
adata.obs["macrostates_fwd"].unique()

[NaN, 'MEF', 'mESC', 'Day 2 Control', 'Day 6 Hic2 OX', 'Day 12 Control']
Categories (5, object): ['MEF', 'Day 2 Control', 'Day 12 Control', 'Day 6 Hic2 OX', 'mESC']

In [20]:
scanpy.tl.rank_genes_groups(adata, groupby='macrostates_fwd', method='wilcoxon')

In [21]:
# 1. Select a cluster (e.g., cluster '0')
cluster_id = 'Day 6 Hic2 OX'

# 2. Extract significant genes only (e.g., adjusted p-value < 0.05)
dedf = scanpy.get.rank_genes_groups_df(adata, group=cluster_id)
gene_list = dedf[dedf['pvals_adj'] < 0.05]['names'].tolist()

# 3. Run Enrichr (ORA)
enr = gseapy.enrichr(
    gene_list=gene_list,
    gene_sets='GO_Biological_Process_2023',
    organism='mouse', # 'human', 'mouse', etc.
    outdir=None       # Don't write to disk
)

# 4. View results
enr.res2d.sort_values('Adjusted P-value').head(20)

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
0,GO_Biological_Process_2023,Cytoplasmic Translation (GO:0002181),8/93,0.000020,0.034007,0,0,7.745213,83.831628,RPS7;RPS18;RPL12;MRPS12;RPL23A;RPL26;RPS10;RPL28
1,GO_Biological_Process_2023,Regulation Of Transforming Growth Factor Beta ...,8/111,0.000071,0.040623,0,0,6.385831,60.967376,BRMS1;CITED2;PEG10;CD109;PMEPA1;TET1;RBBP7;FOLR1
2,GO_Biological_Process_2023,Negative Regulation Of Transmembrane Receptor ...,8/111,0.000071,0.040623,0,0,6.385831,60.967376,BRMS1;PEG10;CD109;PMEPA1;TET1;GDF3;RBBP7;SKIL
3,GO_Biological_Process_2023,Regulation Of Cell Migration (GO:0030334),15/434,0.000346,0.129950,0,0,2.983396,23.771217,ANXA1;CITED2;STAT3;PDGFA;TET1;LAMB1;PLET1;IGF1...
4,GO_Biological_Process_2023,Negative Regulation Of Transforming Growth Fac...,6/77,0.000381,0.129950,0,0,6.901525,54.340290,BRMS1;PEG10;CD109;PMEPA1;TET1;RBBP7
16,GO_Biological_Process_2023,Positive Regulation Of Vesicle Fusion (GO:0031...,2/5,0.001482,0.148832,0,0,53.741497,350.083902,ANXA1;ANXA2
15,GO_Biological_Process_2023,Positive Regulation Of Intrinsic Apoptotic Sig...,2/5,0.001482,0.148832,0,0,53.741497,350.083902,RPS7;RPL26
13,GO_Biological_Process_2023,Negative Regulation Of Cell Motility (GO:2000146),7/133,0.001339,0.148832,0,0,4.543287,30.056202,SFRP1;BRMS1;CITED2;TET1;SPINT2;RBBP7;ARPIN
12,GO_Biological_Process_2023,Epithelial Cell Differentiation (GO:0030855),7/132,0.001282,0.148832,0,0,4.579867,30.498352,UPK1B;TAGLN;ANXA1;BASP1;TAGLN2;GLI1;KRT20
11,GO_Biological_Process_2023,Regulation Of Apoptotic Process (GO:0042981),19/705,0.001263,0.148832,0,0,2.316205,15.458949,MTCH1;BEX1;ANXA1;GADD45B;RIPK3;CITED2;PLAUR;HS...


In [16]:
# 1. Select a cluster (e.g., cluster '0')
cluster_id = 'Day 12 Control'

# 2. Extract significant genes only (e.g., adjusted p-value < 0.05)
dedf = scanpy.get.rank_genes_groups_df(adata, group=cluster_id)
gene_list = dedf[dedf['pvals_adj'] < 0.05]['names'].tolist()

# 3. Run Enrichr (ORA)
enr = gseapy.enrichr(
    gene_list=gene_list,
    gene_sets='GO_Biological_Process_2023',
    organism='mouse', # 'human', 'mouse', etc.
    outdir=None       # Don't write to disk
)

# 4. View results
enr.res2d.sort_values('Adjusted P-value').head(20)

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes
0,GO_Biological_Process_2023,Negative Regulation Of Apoptotic Process (GO:0...,44/482,6.091678e-12,1.675212e-08,0,0,3.699373,95.532981,ITGB1;NOTCH1;PRKAA2;TFRC;ARL6IP1;CITED2;LEF1;F...
1,GO_Biological_Process_2023,Regulation Of Apoptotic Process (GO:0042981),54/705,2.198052e-11,3.022321e-08,0,0,3.080108,75.588509,ITGB1;TOP2A;HIP1;RTKN;TFRC;ARL6IP1;CITED2;FHL2...
2,GO_Biological_Process_2023,Negative Regulation Of Programmed Cell Death (...,37/381,5.286951e-11,4.846371e-08,0,0,3.927209,92.930307,PRKAA2;ARL6IP1;TFRC;CITED2;LEF1;FHL2;HSPB1;FST...
3,GO_Biological_Process_2023,Regulation Of Cell Population Proliferation (G...,55/766,1.642621e-10,1.129302e-07,0,0,2.868906,64.635180,COL18A1;BTG2;CDKN1A;IFITM1;BTG1;TFRC;CD81;HMGB...
4,GO_Biological_Process_2023,Supramolecular Fiber Organization (GO:0097435),30/316,6.181530e-09,3.399841e-06,0,0,3.790870,71.653879,COL18A1;HIP1;SH3KBP1;AVIL;STMN4;KRT20;LOXL2;CS...
5,GO_Biological_Process_2023,Positive Regulation Of Cell Population Prolife...,38/483,1.130035e-08,5.179325e-06,0,0,3.107366,56.859923,CDKN1A;NOTCH1;CD81;LEF1;PDGFA;PTN;LAMC1;THBS1;...
6,GO_Biological_Process_2023,Regulation Of Cell Migration (GO:0030334),35/434,2.339911e-08,9.192508e-06,0,0,3.181454,55.899947,ITGB1;BEX4;IFITM1;NOTCH1;SERPINE2;CITED2;LEF1;...
7,GO_Biological_Process_2023,Positive Regulation Of Cell Differentiation (G...,27/283,3.218539e-08,1.106373e-05,0,0,3.796083,65.489095,FBN2;IFITM1;BTG1;LEF1;TWIST1;PTN;LAMC1;NID1;AC...
9,GO_Biological_Process_2023,Positive Regulation Of Epithelial To Mesenchym...,10/47,5.570147e-07,1.531790e-04,0,0,9.534644,137.305299,COL1A1;TCF7L2;NOTCH1;GLIPR2;MDK;LEF1;TWIST1;EM...
8,GO_Biological_Process_2023,Negative Regulation Of Cellular Process (GO:00...,37/537,5.051142e-07,1.531790e-04,0,0,2.679847,38.853712,COL18A1;BTG2;CDKN1A;IFITM1;BTG1;NOTCH1;JADE1;S...
